In [1]:
import pandas as pd

cutoff_date = pd.Timestamp("2024-09-30")
prediction_end = cutoff_date + pd.Timedelta(days=90)

print("Cutoff:", cutoff_date.date())
print("Prediction end:", prediction_end.date())

Cutoff: 2024-09-30
Prediction end: 2024-12-29


In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/account_features.csv")
df.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,tenure_days,current_mrr,total_usage_count,ticket_count,avg_satisfaction,escalation_rate,has_upgraded,has_downgraded
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,76,12603,535,2.0,3.000000,0.000000,True,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,502,10004,355,3.0,4.000000,0.000000,True,False
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,126,13311,821,3.0,4.666667,0.000000,True,True
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,492,9275,382,2.0,NaN,0.000000,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,65,25355,579,7.0,3.800000,0.142857,True,False


In [2]:
print(df.shape)
df.head()

(500, 18)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,tenure_days,current_mrr,total_usage_count,ticket_count,avg_satisfaction,escalation_rate,has_upgraded,has_downgraded
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,76,12603,535,2.0,3.000000,0.000000,True,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,502,10004,355,3.0,4.000000,0.000000,True,False
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,126,13311,821,3.0,4.666667,0.000000,True,True
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,492,9275,382,2.0,NaN,0.000000,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,65,25355,579,7.0,3.800000,0.142857,True,False


In [2]:
accounts = pd.read_csv("../data/raw/ravenstack_accounts.csv")
subscriptions = pd.read_csv("../data/raw/ravenstack_subscriptions.csv")
usage = pd.read_csv("../data/raw/ravenstack_feature_usage.csv")
support = pd.read_csv("../data/raw/ravenstack_support_tickets.csv")
churn_events = pd.read_csv("../data/raw/ravenstack_churn_events.csv")

print("Accounts:", accounts.shape)
print("Subscriptions:", subscriptions.shape)
print("Usage:", usage.shape)
print("Support:", support.shape)
print("Churn events:", churn_events.shape)

Accounts: (500, 10)
Subscriptions: (5000, 14)
Usage: (25000, 8)
Support: (2000, 9)
Churn events: (600, 9)


In [3]:
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])

subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])

usage["usage_date"] = pd.to_datetime(usage["usage_date"])

support["submitted_at"] = pd.to_datetime(support["submitted_at"])

churn_events["churn_date"] = pd.to_datetime(churn_events["churn_date"])

In [4]:
future_churns = churn_events[
    (churn_events["churn_date"] > cutoff_date) &
    (churn_events["churn_date"] <= prediction_end)
]

churned_accounts = set(future_churns["account_id"])

accounts["churn_target"] = accounts["account_id"].isin(
    churned_accounts
).astype(int)

print("Target distribution:")
print(accounts["churn_target"].value_counts())

print("\nChurn rate:")
print(accounts["churn_target"].mean())

Target distribution:
churn_target
0    325
1    175
Name: count, dtype: int64

Churn rate:
0.35


In [6]:
eligible_accounts = accounts[
    accounts["signup_date"] <= cutoff_date
].copy()

account_features = eligible_accounts[
    [
        "account_id",
        "account_name",
        "industry",
        "country",
        "referral_source",
        "plan_tier",
        "seats",
        "is_trial"
    ]
].copy()

account_features["tenure_days"] = (
    cutoff_date - eligible_accounts["signup_date"]
).dt.days

print("Eligible accounts:", len(account_features))
print("Minimum tenure:", account_features["tenure_days"].min())
print("Maximum tenure:", account_features["tenure_days"].max())

Eligible accounts: 420
Minimum tenure: 1
Maximum tenure: 637


In [7]:
active_subscriptions = subscriptions[
    (subscriptions["start_date"] <= cutoff_date) &
    (
        subscriptions["end_date"].isna() |
        (subscriptions["end_date"] >= cutoff_date)
    )
].copy()

print("Active subscriptions at cutoff:", len(active_subscriptions))
print(active_subscriptions[
    ["account_id", "plan_tier", "seats", "mrr_amount", "arr_amount"]
].head())

Active subscriptions at cutoff: 2770
   account_id   plan_tier  seats  mrr_amount  arr_amount
1    A-9b9fe9         Pro     17         833        9996
4    A-ba6516  Enterprise     27        5373       64476
5    A-fa2041         Pro     15         735        8820
6    A-417d2f  Enterprise      4         796        9552
10   A-f03140         Pro     27        1323       15876


In [8]:
revenue_features = (
    active_subscriptions
    .groupby("account_id")
    .agg(
        current_mrr=("mrr_amount", "sum"),
        current_arr=("arr_amount", "sum"),
        active_seats=("seats", "sum")
    )
    .reset_index()
)

print(revenue_features.shape)
print(revenue_features.head())

(415, 4)
  account_id  current_mrr  current_arr  active_seats
0   A-00bed1        33506       402072           342
1   A-00cac8         9210       110520            99
2   A-0158bb          836        10032            69
3   A-016043         4341        52092            51
4   A-019782         5169        62028            92


In [9]:
account_features = account_features.merge(
    revenue_features,
    on="account_id",
    how="left"
)

account_features[
    ["current_mrr", "current_arr", "active_seats"]
] = account_features[
    ["current_mrr", "current_arr", "active_seats"]
].fillna(0)

print(account_features.shape)
print(account_features[
    ["account_id", "current_mrr", "current_arr", "active_seats"]
].head())

(420, 12)
  account_id  current_mrr  current_arr  active_seats
0   A-43a9e3      10004.0     120048.0         176.0
1   A-0a282f       1194.0      14328.0           6.0
2   A-1f0ac7       7491.0      89892.0         153.0
3   A-1b9609      16614.0     199368.0         186.0
4   A-a0ca4e      10920.0     131040.0         120.0


In [10]:
usage_before_cutoff = usage[
    usage["usage_date"] <= cutoff_date
].copy()

usage_before_cutoff = usage_before_cutoff.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

print("Usage records before/on cutoff:", len(usage_before_cutoff))
print(usage_before_cutoff.head())

Usage records before/on cutoff: 21796
   usage_id subscription_id usage_date feature_name  usage_count  \
0  U-1c6c24        S-0fcf7d 2023-07-27   feature_20            9   
1  U-f07cb8        S-c25263 2023-08-07    feature_5            9   
2  U-096807        S-f29e7f 2023-12-07    feature_3            9   
3  U-6b1580        S-be655e 2024-07-28   feature_40            5   
4  U-c5692d        S-e958ba 2023-10-31   feature_14           11   

   usage_duration_secs  error_count  is_beta_feature account_id  
0                 5004            0            False   A-e08cd3  
1                  369            0            False   A-c7ffc2  
2                 1458            0            False   A-bbe56f  
3                 2085            0            False   A-7f29a7  
4                 4103            0            False   A-417d2f  


In [11]:
usage_features = (
    usage_before_cutoff
    .groupby("account_id")
    .agg(
        total_usage_count=("usage_count", "sum")
    )
    .reset_index()
)

print(usage_features.shape)
print(usage_features.head())

(500, 2)
  account_id  total_usage_count
0   A-00bed1                462
1   A-00cac8                555
2   A-0158bb                339
3   A-016043                427
4   A-019782                498


In [12]:
account_features = account_features.merge(
    usage_features,
    on="account_id",
    how="left"
)

account_features["total_usage_count"] = (
    account_features["total_usage_count"].fillna(0)
)

print(account_features.shape)
print(
    account_features[
        ["account_id", "total_usage_count"]
    ].head()
)

(420, 13)
  account_id  total_usage_count
0   A-43a9e3                317
1   A-0a282f                717
2   A-1f0ac7                337
3   A-1b9609                390
4   A-a0ca4e                311


In [13]:
support_before_cutoff = support[
    support["submitted_at"] <= cutoff_date
].copy()

print("Support records before/on cutoff:", len(support_before_cutoff))
print(support_before_cutoff.head())

Support records before/on cutoff: 1730
  ticket_id account_id submitted_at            closed_at  \
0  T-0024de   A-712f1c   2023-07-27  2023-07-28 03:00:00   
1  T-4d04b9   A-e43bf7   2024-07-08  2024-07-09 03:00:00   
3  T-dfce9a   A-4c56c9   2024-09-08  2024-09-09 23:00:00   
5  T-90f06d   A-94c3cd   2023-07-27  2023-07-27 09:00:00   
6  T-30b537   A-2e4581   2023-09-09  2023-09-10 03:00:00   

   resolution_time_hours priority  first_response_time_minutes  \
0                   27.0     high                           74   
1                   27.0   urgent                          144   
3                   47.0   medium                          126   
5                    9.0   medium                           60   
6                   27.0   urgent                           64   

   satisfaction_score  escalation_flag  
0                 NaN            False  
1                 NaN            False  
3                 5.0            False  
5                 NaN            False 

In [14]:
support_features = (
    support_before_cutoff
    .groupby("account_id")
    .agg(
        ticket_count=("ticket_id", "count"),
        avg_satisfaction=("satisfaction_score", "mean"),
        escalation_rate=("escalation_flag", "mean")
    )
    .reset_index()
)

print(support_features.shape)
print(support_features.head())

(490, 4)
  account_id  ticket_count  avg_satisfaction  escalation_rate
0   A-00bed1             4               4.0              0.0
1   A-00cac8             2               NaN              0.0
2   A-0158bb             1               3.0              0.0
3   A-016043             2               4.0              0.0
4   A-019782             1               NaN              0.0


In [15]:
account_features = account_features.merge(
    support_features,
    on="account_id",
    how="left"
)

account_features["ticket_count"] = (
    account_features["ticket_count"].fillna(0)
)

account_features["escalation_rate"] = (
    account_features["escalation_rate"].fillna(0)
)

print(account_features.shape)
print(
    account_features[
        ["account_id", "ticket_count",
         "avg_satisfaction", "escalation_rate"]
    ].head()
)

(420, 16)
  account_id  ticket_count  avg_satisfaction  escalation_rate
0   A-43a9e3           3.0               4.0              0.0
1   A-0a282f           2.0               5.0              0.0
2   A-1f0ac7           2.0               NaN              0.0
3   A-1b9609           3.0               3.0              0.0
4   A-a0ca4e           5.0               3.8              0.6


In [16]:
subscriptions_before_cutoff = subscriptions[
    subscriptions["start_date"] <= cutoff_date
].copy()

behavior_features = (
    subscriptions_before_cutoff
    .groupby("account_id")
    .agg(
        has_upgraded=("upgrade_flag", "max"),
        has_downgraded=("downgrade_flag", "max")
    )
    .reset_index()
)

print(behavior_features.shape)
print(behavior_features.head())

(416, 3)
  account_id  has_upgraded  has_downgraded
0   A-00bed1          True           False
1   A-00cac8         False           False
2   A-0158bb         False           False
3   A-016043          True           False
4   A-019782          True           False


In [17]:
account_features = account_features.merge(
    behavior_features,
    on="account_id",
    how="left"
)

account_features["has_upgraded"] = (
    account_features["has_upgraded"].fillna(False)
)

account_features["has_downgraded"] = (
    account_features["has_downgraded"].fillna(False)
)

print(account_features.shape)
print(account_features.columns.tolist())

(420, 18)
['account_id', 'account_name', 'industry', 'country', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'tenure_days', 'current_mrr', 'current_arr', 'active_seats', 'total_usage_count', 'ticket_count', 'avg_satisfaction', 'escalation_rate', 'has_upgraded', 'has_downgraded']


In [18]:
account_features = account_features.merge(
    accounts[["account_id", "churn_target"]],
    on="account_id",
    how="left"
)

print("Final shape:", account_features.shape)

print("\nTarget distribution:")
print(account_features["churn_target"].value_counts())

print("\nMissing values:")
print(account_features.isna().sum())

Final shape: (420, 19)

Target distribution:
churn_target
0    296
1    124
Name: count, dtype: int64

Missing values:
account_id            0
account_name          0
industry              0
country               0
referral_source       0
plan_tier             0
seats                 0
is_trial              0
tenure_days           0
current_mrr           0
current_arr           0
active_seats          0
total_usage_count     0
ticket_count          0
avg_satisfaction     44
escalation_rate       0
has_upgraded          0
has_downgraded        0
churn_target          0
dtype: int64


In [19]:
account_features.to_csv(
    "../data/processed/temporal_account_features.csv",
    index=False
)

print("Saved temporal feature dataset!")

Saved temporal feature dataset!


In [3]:
df.columns.tolist()

['account_id',
 'account_name',
 'industry',
 'country',
 'signup_date',
 'referral_source',
 'plan_tier',
 'seats',
 'is_trial',
 'churn_flag',
 'tenure_days',
 'current_mrr',
 'total_usage_count',
 'ticket_count',
 'avg_satisfaction',
 'escalation_rate',
 'has_upgraded',
 'has_downgraded']